# 百度地图地铁数据处理


In [1]:
import pandas as pd
import numpy as np
import json
from collections import defaultdict

import networkx as nx

In [2]:
import sys
sys.path.append("../../..")
from secure.db_account import SubwayPrd
from k_libs.db_query import DBOperate

DBO = DBOperate(SubwayPrd)

## 字段表


In [3]:
# 线路字段
line_fields = [
    {'raw': 'lid',
    'processed': 'line_name_full',
    'info': '线路全名'},
    {'raw': 'lb',
    'processed': 'line_name',
    'info': '线路简称'},
    {'raw': 'n',
    'processed': 'num_stations',
    'info': '线路站点数'},
    {'raw': 'loop',
    'processed': 'is_loop',
    'info': '是否环线'},
    {'raw': 'lbx',
    'processed': 'label_x',
    'info': '线路标签X坐标'},
    {'raw': 'lby',
    'processed': 'label_y',
    'info': '线路标签Y坐标'},
    {'raw': 'lbr',
    'processed': 'label_rotation',
    'info': '线路标签旋转角度'},
    {'raw': 'lc',
    'processed': 'line_color',
    'info': '线路颜色'},
    {'raw': 'uid',
    'processed': 'line_uid',
    'info': '线路唯一ID'},
    {'raw': 'uid2',
    'processed': 'line_uid2',
    'info': '线路备用唯一ID'}
]

In [4]:
# 站点字段
station_fields = [
    {'raw': 'sid',
    'processed': 'station_name',
    'info': '站点名称'},
    {'raw': 'lb',
    'processed': 'station_label',
    'info': '站点标签'},
    {'raw': 'x',
    'processed': 'x',
    'info': '站点X坐标'},
    {'raw': 'y',
    'processed': 'y',
    'info': '站点Y坐标'},
    {'raw': 'rx',
    'processed': 'label_offset_x',
    'info': '站点标签X偏移'},
    {'raw': 'ry',
    'processed': 'label_offset_y',
    'info': '站点标签Y偏移'},
    {'raw': 'st',
    'processed': 'is_station',
    'info': '是否为车站'},
    {'raw': 'ex',
    'processed': 'is_exchange',
    'info': '是否换乘点'},
    {'raw': 'iu',
    'processed': 'is_use',
    'info': '是否在用,true表示该站已开通并运营'},
    {'raw': 'rc',
    'processed': 'is_rail_construction',
    'info': '是否为规划/在建站点'},
    {'raw': 'slb',
    'processed': 'show_label',
    'info': '是否显示标签'},
    {'raw': 'ln',
    'processed': 'lines',
    'info': '所属线路'},
    {'raw': 'uid',
    'processed': 'station_uid',
    'info': '站点唯一ID'},
    {'raw': 'px',
    'processed': 'proj_x',
    'info': '站点投影坐标X'},
    {'raw': 'py',
    'processed': 'proj_y',
    'info': '站点投影坐标Y'}
    ]


## 原始数据
ODS 层（Operational Data Store，操作数据存储层）

存放业务系统的原始数据，基本不做加工。

特点：与业务库字段保持一致，保证数据的完整性与可追溯性。

作用：承接源系统，作为数据仓库的“原材料”。

In [5]:
sql = """SELECT * FROM ods_subway_baidu WHERE crawler_id=(SELECT MAX(crawler_id) FROM ods_subway_baidu);"""
df_raw = DBO.read_sql(sql)

In [6]:
df_raw[df_raw['line_lb'] == '郑许线']

,line_lid,line_lb,line_slb,line_n,line_loop,line_lbx,line_lby,line_lbr,line_lc,line_uid,...,st_int,st_uid,st_px,st_py,city_id,city_name,city_name_e,crawler_id,crawler_date,uid
9294,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,cabad1ce2fa272b99d9d68b1,12674212.92,4085517.41,268,郑州,zhengzhou,251230,2025-12-30,268251230331
9295,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,5794ce530858800fc8213d02,12673051.66,4083999.48,268,郑州,zhengzhou,251230,2025-12-30,268251230332
9296,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,,0,0,268,郑州,zhengzhou,251230,2025-12-30,268251230333
9297,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,3978a5fc93cb6a3ce81e976c,12672706.74,4079926.41,268,郑州,zhengzhou,251230,2025-12-30,268251230334
9298,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,35beb72740a0786be9f853b7,12672393.37,4078500.76,268,郑州,zhengzhou,251230,2025-12-30,268251230335
9299,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,0b59d369f8d3b54d6da3cc5a,12672581.26,4076936.97,268,郑州,zhengzhou,251230,2025-12-30,268251230336
9300,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,2,db2af5362cbbff9f3686f47d,12674501.81,4075968.39,268,郑州,zhengzhou,251230,2025-12-30,268251230337
9301,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,None,None,None,None,268,郑州,zhengzhou,251230,2025-12-30,268251230338
9302,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,None,None,None,None,268,郑州,zhengzhou,251230,2025-12-30,268251230339
9303,郑许线,郑许线,False,26,False,1540,1270,0,0x243F4E,d555e4bf2a72d94d260bd084,...,None,None,None,None,268,郑州,zhengzhou,251230,2025-12-30,268251230340


## DWD 
DWD 层（Data Warehouse Detail，明细数据层）  
在 ODS 的基础上进行清洗、规范化，保留明细粒度的数据。  
特点：字段标准化（统一命名、数据类型），做一些维度退化或拆分。  
作用：保证数据“可用”，是最常被下游加工使用的一层。  

In [7]:
def dwd_subway_bd_to_sql(df_raw):
    """
    将百度地铁数据处理为DWD层数据，并存入数据库
    """

    # 对字段重命名
    df_dwd = df_raw.copy()
    df_dwd = df_dwd.rename(
        columns={
            'line_lid': 'line_name_full',
            'line_lb': 'line_name',
            'line_slb': 'line_show_label',
            'line_loop': 'line_is_loop',
            'line_lbx': 'line_label_x',
            'line_lby': 'line_label_y',
            'line_lbr': 'line_label_rotation',
            'line_lc': 'line_color',
            'st_sid': 'st_name',
            'st_lb': 'st_label',
            'st_rx': 'st_label_offset_x',
            'st_ry': 'st_label_offset_y',
            'st_st': 'st_is_station',
            'st_ex': 'st_is_exchange',
            'st_iu': 'st_is_use',
            'st_rc': 'st_is_rail_construction',
            'st_slb': 'st_show_label',
            'st_ln': 'st_lines',
            'st_px': 'st_proj_x',
            'st_py': 'st_proj_y'
        })
    # 字段排序
    field_order = [
        'city_id',
        'city_name',
        'city_name_e',
        # 线路信息
        'line_name_full',
        'line_name',
        'line_show_label',
        'line_n',
        'line_is_loop',
        'line_label_x',
        'line_label_y',
        'line_label_rotation',
        'line_color',
        'line_uid',
        'line_uid2',
        # 站点信息
        'st_name',
        'st_label',
        'st_x',
        'st_y',
        'st_label_offset_x',
        'st_label_offset_y',
        'st_is_station',
        'st_is_exchange',
        'st_is_use',
        'st_is_rail_construction',
        'st_show_label',
        'st_lines',
        'st_int',
        'st_uid',
        'st_proj_x',
        'st_proj_y',
        # 数据采集信息
        'crawler_id',
        'crawler_date',
        'uid'
    ]
    df_dwd = df_dwd[field_order]
    # 对存在false,true的字段进行处理
    bool_fields = [
        'line_show_label', 'line_is_loop', 'st_is_station', 'st_is_exchange',
        'st_is_use', 'st_is_rail_construction', 'st_show_label'
    ]
    df_dwd = df_dwd.copy()
    for field in bool_fields:
        df_dwd[field] = df_dwd[field].map({'True': 1, 'False': 0})
    # 颜色修改为16进制
    df_dwd['line_color'] = df_dwd['line_color'].apply(
        lambda x: f"#{x[2:]}" if not x.startswith('#') else x)
    # 判断是否为车站,st_name是否为空
    df_dwd['st_is_station'] = df_dwd['st_name'].apply(lambda x: 1
                                                      if len(x) > 0 else 0)
    # 添加线路id
    df_dwd['line_name_merged'] = df_dwd['line_name_full'] + df_dwd['line_name']
    df_dwd['line_id_k'] = df_dwd.groupby('city_id')[
        'line_name_merged'].transform(lambda x: pd.factorize(x)[0] + 1)
    df_dwd['line_uid_k'] = df_dwd.apply(
        lambda row: f"{row['city_id']}_{row['line_id_k']:03d}", axis=1)
    # 郑州特殊处理，保存一份不含郑许线的郑州数据，城市名为郑州s，城市id为268s
    df_zz = df_dwd[df_dwd['city_name'] == '郑州'].copy()
    df_zz = df_zz[~df_zz['st_lines'].str.contains('郑许线')]
    df_zz['city_name'] = '郑州s'
    df_zz['city_id'] = '268s'
    df_dwd = pd.concat([df_dwd, df_zz], ignore_index=True)

    # 保存到数据库
    DBO.df_to_sql(df_dwd, "dwd_subway_baidu", if_exists="replace")
    print("Data saved to dwd_subway_baidu table.")
    return df_dwd

dwd_subway = dwd_subway_bd_to_sql(df_raw)
dwd_subway.info()

Data saved to dwd_subway_baidu table.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18737 entries, 0 to 18736
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   city_id                  18737 non-null  object 
 1   city_name                18737 non-null  object 
 2   city_name_e              18737 non-null  object 
 3   line_name_full           18737 non-null  object 
 4   line_name                18737 non-null  object 
 5   line_show_label          18737 non-null  int64  
 6   line_n                   18737 non-null  object 
 7   line_is_loop             18737 non-null  int64  
 8   line_label_x             18737 non-null  object 
 9   line_label_y             18737 non-null  object 
 10  line_label_rotation      18737 non-null  object 
 11  line_color               18737 non-null  object 
 12  line_uid                 18737 non-null  object 
 13  line_uid2                18737 non-nul

In [8]:
dwd_subway

,city_id,city_name,city_name_e,line_name_full,line_name,line_show_label,line_n,line_is_loop,line_label_x,line_label_y,...,st_int,st_uid,st_proj_x,st_proj_y,crawler_id,crawler_date,uid,line_name_merged,line_id_k,line_uid_k
0,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,3,291c5802f26a751cbca240d9,12935140.04,4825694.5,251230,2025-12-30,1312512300,地铁1号线八通线1号线八通线,1,131_001
1,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,3,ad28546df35285eb851541d9,12937624.6,4825645.68,251230,2025-12-30,1312512301,地铁1号线八通线1号线八通线,1,131_001
2,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,3,a047555503a5bc5cbc4842d9,12940185.58,4825661.64,251230,2025-12-30,1312512302,地铁1号线八通线1号线八通线,1,131_001
3,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,3,a27fa5a23a128501177643d9,12942086.4,4825707.12,251230,2025-12-30,1312512303,地铁1号线八通线1号线八通线,1,131_001
4,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,3,53889c15034f2e3f58bbbcde,12944444.84,4825755.15,251230,2025-12-30,1312512304,地铁1号线八通线1号线八通线,1,131_001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18732,268s,郑州s,zhengzhou,郑许线,郑许线,0,26,0,1540,1270,...,None,None,None,None,251230,2025-12-30,268251230339,郑许线郑许线,13,268_013
18733,268s,郑州s,zhengzhou,郑许线,郑许线,0,26,0,1540,1270,...,None,None,None,None,251230,2025-12-30,268251230340,郑许线郑许线,13,268_013
18734,268s,郑州s,zhengzhou,郑许线,郑许线,0,26,0,1540,1270,...,None,None,None,None,251230,2025-12-30,268251230342,郑许线郑许线,13,268_013
18735,268s,郑州s,zhengzhou,郑许线,郑许线,0,26,0,1540,1270,...,None,None,None,None,251230,2025-12-30,268251230343,郑许线郑许线,13,268_013


In [9]:
DBO = DBOperate(SubwayPrd)
def query_city_infos():
    """
    查询城市信息
    """
    sql = "SELECT * FROM dim_city_info;"
    df_city = DBO.read_sql(sql)
    return df_city
dwd_city_info = query_city_infos()
dwd_city_info

,city_name,english_pinyin_name,province_chinese,province_pinyin_english,country_chinese,country_english,note
0,北京,Beijing,北京市,Beijing Shi,中国,China,None
1,上海,Shanghai,上海市,Shanghai Shi,中国,China,None
2,广州,Guangzhou,广东省,Guangdong Sheng,中国,China,None
3,深圳,Shenzhen,广东省,Guangdong Sheng,中国,China,None
4,重庆,Chongqing,重庆市,Chongqing Shi,中国,China,None
...,...,...,...,...,...,...,...
75,莫斯科,Moscow,莫斯科州,Moscow Oblast,俄罗斯,Russia,None
76,鹿特丹,Rotterdam,南荷兰省,South Holland Province,荷兰,Netherlands,None
77,伊斯坦布尔,Istanbul,伊斯坦布尔省,Istanbul Province,土耳其,Turkey,None
78,巴塞罗那,Barcelona,加泰罗尼亚自治区,Catalonia Autonomous Community,西班牙,Spain,None


## DWS
DWS 层（Data Warehouse Summary，汇总数据层）

在 DWD 基础上，按业务主题和常用维度做聚合统计。

特点：以主题域为核心（如用户、订单、交易），形成宽表或统计指标。

作用：减少重复计算，支撑公共分析需求。

1. 城市表 dws_subway_bd_city
2. 线路表 dws_subway_bd_line
3. 车站表 dws_subway_bd_st

### 城市表

In [10]:
def get_city_infos_dws(df, city_info):
    """
    生成城市信息表
    df: DWD层数据, dwd_subway
    city_info: 城市信息字典
    """
    # df_city = df[['city_id', 'city_name', 'city_name_e']].drop_duplicates().reset_index(drop=True)
    df_city = df.groupby(['city_id', 'city_name'])['line_name_full'].nunique().reset_index(name='line_count')
    # city_order， 按线路数排序后，
    df_city = df_city.sort_values(by='line_count', ascending=False).reset_index(drop=True)

    df_city['city_order'] = df_city.index + 1
    df_city = df_city.merge(city_info, how='left', on='city_name')
    df_city = df_city.rename(columns={'english_pinyin_name': 'city_name_e',
        'province_chinese': 'province', 
                                    'province_pinyin_english': 'province_e',
                                      'country_chinese': 'country', 
                                      'country_english': 'country_e'})
    return df_city
dws_subway_city = get_city_infos_dws(dwd_subway, dwd_city_info)
dws_subway_city

,city_id,city_name,line_count,city_order,city_name_e,province,province_e,country,country_e,note
0,60732,纽约,32,1,New York,纽约州,New York State,美国,United States,None
1,30016,首尔,30,2,Seoul,首尔特别市,Seoul Special City,韩国,South Korea,None
2,257,广州,29,3,Guangzhou,广东省,Guangdong Sheng,中国,China,None
3,131,北京,28,4,Beijing,北京市,Beijing Shi,中国,China,None
4,289,上海,25,5,Shanghai,上海市,Shanghai Shi,中国,China,None
...,...,...,...,...,...,...,...,...,...,...
75,155,许昌,1,76,Xuchang,河南省,Henan Sheng,中国,China,None
76,30007,光州,1,77,Gwangju,光州广域市,Gwangju Metropolitan City,韩国,South Korea,None
77,313,湘潭,1,78,Xiangtan,湖南省,Hunan Sheng,中国,China,None
78,122,鄂州,1,79,Ezhou,湖北省,Hubei Sheng,中国,China,None


### 线路表
* line_uid有空值，不可用
* line_name_full 暂无重复值，可以用
* line_name 有重复值，需处理

#### 支线名称清洗
对于line_name_full 不唯一，line_name唯一的线路，将line_name_full更新为line_name

In [11]:
def clear_line_branch(df):
    """
    清理线路支线信息，将line_name_full作为支线名称，支线名称应该唯一
    df: dwd_subway
    """
    df_line = df[[
        'city_id', 'city_name', 'line_name_full', 'line_name', 'line_is_loop',
        'line_label_x', 'line_label_y', 'line_label_rotation', 'line_color', 'line_uid_k'
    ]].drop_duplicates().reset_index(drop=True)
    # 按city_name, line_name统计line_name出现次数
    df_line['line_name_count'] = df_line.groupby(
        ['city_name', 'line_name'])['line_name'].transform('count')
    # if df_line['line_name_count'].max() > 1:
    #     print("存在line_name相同的线路")
    #     print(df_line[df_line['line_name_count'] > 1])
    # 按city_name, line_name_full统计line_name出现次数
    df_line['line_name_full_count'] = df_line.groupby(
        ['city_name', 'line_name_full'])['line_name_full'].transform('count')
    print(df_line[df_line['line_name_full_count'] > 1])
    if df_line['line_name_full_count'].max() > 1:
        # 停止执行，查看后可注释掉，继续处理
        # raise ValueError("存在line_name_full相同的线路，非唯一，需要定位数据，做进一步处理")
        # 对于line_name_full 不唯一，line_name唯一的线路，将line_name_full更新为line_name
        for idx, row in df_line[df_line['line_name_full_count'] >
                                1].iterrows():
            df_line.at[idx, 'line_name_full'] = row['line_name']
    # 添加line_name_branch，当line_name_count>1时，使用line_name_full
    df_line['line_name_branch'] = df_line['line_name']
    df_line.loc[df_line['line_name_count'] > 1,
                'line_name_branch'] = df_line['line_name_full']
    # 删除line_name_full中的“地铁”
    df_line['line_name_branch'] = df_line['line_name_branch'].str.replace(
        '地铁', '', regex=False)
    # 按城市，线路顺序添加line_order
    df_line['line_order'] = df_line.groupby('city_id').cumcount() + 1
    return df_line

df_line_branch = clear_line_branch(dwd_subway)
df_line_branch

   city_id city_name line_name_full            line_name  line_is_loop  \
55     257        广州          地铁3号线        3号线(海傍-天河客运站)             0   
56     257        广州          地铁3号线  3号线(体育西路-机场北(2号航站楼)             0   
65     257        广州         地铁12号线      12号线(浔峰岗-广州体育馆)             0   
66     257        广州         地铁12号线       12号线(二沙岛-大学城南)             0   
74     257        广州        佛山地铁3号线  佛山地铁3号线(中山公园-顺德学院站)             0   
75     257        广州        佛山地铁3号线     佛山地铁3号线(联和-佛山大学)             0   

   line_label_x line_label_y line_label_rotation line_color line_uid_k  \
55          230         -300                   0    #EEB56E    257_003   
56          -60         -530                   0    #EEB56E    257_004   
65         -470         -560                   0    #828757    257_013   
66          320          287                   0    #828757    257_014   
74         -300          840                   0    #456E9B    257_022   
75         -980         -310         

,city_id,city_name,line_name_full,line_name,line_is_loop,line_label_x,line_label_y,line_label_rotation,line_color,line_uid_k,line_name_count,line_name_full_count,line_name_branch,line_order
0,131,北京,地铁1号线八通线,1号线八通线,0,-498.9,139.1,0,#D3746E,131_001,1,1,1号线八通线,1
1,131,北京,地铁2号线,2号线,1,400,-40,0,#568BB2,131_002,1,1,2号线,2
2,131,北京,地铁3号线,3号线,0,1580,-100,0,#F05771,131_003,1,1,3号线,3
3,131,北京,地铁4号线大兴线,4号线大兴线,0,-180,-700,0,#49ACBD,131_004,1,1,4号线大兴线,4
4,131,北京,地铁5号线,5号线,0,860,-1120,0,#C162A1,131_005,1,1,5号线,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
649,268s,郑州s,地铁10号线,10号线,0,-1150,220,0,#CD877F,268_009,1,1,10号线,9
650,268s,郑州s,地铁12号线,12号线,0,1620,-170,0,#578BEF,268_010,1,1,12号线,10
651,268s,郑州s,地铁14号线,14号线,0,-575,570,0,#C5A7CA,268_011,1,1,14号线,11
652,268s,郑州s,城郊线,城郊线,0,520,870,0,#B8C29D,268_012,1,1,城郊线,12


#### 主线名称清洗
1. 计算line_name与上一行相同字数，作为same_prefix_len列
2. 筛选出same_prefix_len>2的行，以及每一的上一行
3. 将['city_id', 'city_name', 'line_name_full', 'line_name', 'line_name_branch']保存为新的df
4. 对比subway_line_fixed.json文件，将新增的手工添加"line_name_main"字段，复制到subway_line_fixed.json文件最后。

In [12]:
def clear_line_main(df_line_branch, subway_line_fixed):
    """
    为支线添加主线名称
    df_line_branch: 支线数据
    subway_line_fixed: 从subway_line_fixed.json中读取的字典数据
    """
    df_line_main = df_line_branch.copy()
    # 按城市，统计line_name与上一行line_name相同的字数，从首字开始统计，如首字不同，直接记为0
    df_line_main['same_prefix_len'] = 0
    for city in df_line_main['city_name'].unique():
        city_mask = df_line_main[df_line_main['city_name'] == city]
        prev_line_name = ""
        for idx, row in city_mask.iterrows():
            line_name = row['line_name']
            if line_name == prev_line_name:
                df_line_main.at[idx, 'same_prefix_len'] = len(line_name)
            else:
                # 计算与上一行相同的前缀长度
                common_length = 0
                for c1, c2 in zip(line_name, prev_line_name):
                    if c1 == c2:
                        common_length += 1
                    else:
                        break
                df_line_main.at[idx, 'same_prefix_len'] = common_length
            prev_line_name = line_name
    # 筛选出same_prefix_len>2的行，以及每一的上一行
    df_line_main_filtered = pd.DataFrame()
    for city in df_line_main['city_name'].unique():
        city_mask = df_line_main[df_line_main['city_name'] == city]
        indices = city_mask.index.tolist()
        for i in range(1, len(indices)):
            if city_mask.at[indices[i], 'same_prefix_len'] > 2:
                df_line_main_filtered = pd.concat([df_line_main_filtered, city_mask.loc[[indices[i-1], indices[i]]]])
    df_line_main_filtered = df_line_main_filtered.drop_duplicates().reset_index(drop=True)

    df_line_main_filtered_dict = df_line_main_filtered[['city_id', 'city_name', 'line_name_full', 'line_name', 'line_name_branch']].to_dict(orient='records')
    # 对比subway_line_fixed文件，打印df_line_main_filtered_dict中每项前4项不在subway_line_fixed中的行
    fields = ['city_id', 'city_name', 'line_name_full', 'line_name']
    fixed_set = set(
        tuple(line[field] for field in fields)
        for line in subway_line_fixed
    )
    # 手动为下方打印的行添加"line_name_main"字段，复制到subway_line_fixed.json文件最后
    # 保存subway_line_fixed.json文件
    print("以下线路需要手动添加主线名称line_name_main，并复制到subway_line_fixed.json文件最后")
    for item in df_line_main_filtered_dict:
        key = tuple(item[field] for field in fields)
        if key not in fixed_set:
            print(item)

with open("subway_line_fixed.json", "r", encoding="utf-8") as f:
    subway_line_fixed = json.load(f)
clear_line_main(df_line_branch, subway_line_fixed)

以下线路需要手动添加主线名称line_name_main，并复制到subway_line_fixed.json文件最后
{'city_id': '179', 'city_name': '杭州', 'line_name_full': '地铁3号线(吴山前村-星桥)', 'line_name': '3号线(吴山前村-星桥)', 'line_name_branch': '3号线(吴山前村-星桥)'}
{'city_id': '179', 'city_name': '杭州', 'line_name_full': '地铁3号线(石马-星桥)', 'line_name': '3号线(石马-星桥)', 'line_name_branch': '3号线(石马-星桥)'}
{'city_id': '179', 'city_name': '杭州', 'line_name_full': '地铁6号线(枸桔弄-桂花西路)', 'line_name': '6号线(枸桔弄-桂花西路)', 'line_name_branch': '6号线(枸桔弄-桂花西路)'}
{'city_id': '179', 'city_name': '杭州', 'line_name_full': '地铁6号线(枸桔弄-双浦)', 'line_name': '6号线(枸桔弄-双浦)', 'line_name_branch': '6号线(枸桔弄-双浦)'}
{'city_id': '138', 'city_name': '佛山', 'line_name_full': '佛山地铁2号线', 'line_name': '佛山地铁2号线', 'line_name_branch': '佛山2号线'}
{'city_id': '138', 'city_name': '佛山', 'line_name_full': '佛山地铁3号线(中山公园-顺德学院站)', 'line_name': '佛山地铁3号线(中山公园-顺德学院站)', 'line_name_branch': '佛山3号线(中山公园-顺德学院站)'}
{'city_id': '138', 'city_name': '佛山', 'line_name_full': '佛山地铁3号线(联和-佛山大学)', 'line_name': '佛山地铁3号线(联和-佛山大学)', 'line

#### 合并线路数据

In [13]:
with open("subway_line_fixed.json", "r", encoding="utf-8") as f:
    subway_line_fixed = json.load(f)
subway_line_fixed_df = pd.DataFrame.from_dict(subway_line_fixed)
dws_subway_bd_line = df_line_branch.merge(subway_line_fixed_df[['city_id', 'line_name_full', 'line_name_main']], how='left', on=['city_id', 'line_name_full']) 
# 将line_name_main中缺失值，填充为line_name_branch
dws_subway_bd_line['line_name_main'] = dws_subway_bd_line['line_name_main'].fillna(dws_subway_bd_line['line_name_branch'])
dws_subway_bd_line


,city_id,city_name,line_name_full,line_name,line_is_loop,line_label_x,line_label_y,line_label_rotation,line_color,line_uid_k,line_name_count,line_name_full_count,line_name_branch,line_order,line_name_main
0,131,北京,地铁1号线八通线,1号线八通线,0,-498.9,139.1,0,#D3746E,131_001,1,1,1号线八通线,1,1号线八通线
1,131,北京,地铁2号线,2号线,1,400,-40,0,#568BB2,131_002,1,1,2号线,2,2号线
2,131,北京,地铁3号线,3号线,0,1580,-100,0,#F05771,131_003,1,1,3号线,3,3号线
3,131,北京,地铁4号线大兴线,4号线大兴线,0,-180,-700,0,#49ACBD,131_004,1,1,4号线大兴线,4,4号线大兴线
4,131,北京,地铁5号线,5号线,0,860,-1120,0,#C162A1,131_005,1,1,5号线,5,5号线
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
652,268s,郑州s,地铁10号线,10号线,0,-1150,220,0,#CD877F,268_009,1,1,10号线,9,10号线
653,268s,郑州s,地铁12号线,12号线,0,1620,-170,0,#578BEF,268_010,1,1,12号线,10,12号线
654,268s,郑州s,地铁14号线,14号线,0,-575,570,0,#C5A7CA,268_011,1,1,14号线,11,14号线
655,268s,郑州s,城郊线,城郊线,0,520,870,0,#B8C29D,268_012,1,1,城郊线,12,城郊线


### 车站表
车站唯一id问题，存在同一个城市有同名车站，比如杭州，有2个奥体中心，1个在杭州，1个在绍兴，需要使用sid（st_name）作为城市唯一标识

In [14]:
def get_dwd_subway_st(df, df_line, is_all_st=True):
    """
    获取地铁站数据
    车站含虚拟车站和真实车站两种
    is_all_st: True-全量车站， False-真实车站
    df: dwd_subway数据
    df_line: dwd_subway_bd_line数据
    """
    df = df.copy() if is_all_st else df[df['st_is_station'] == 1].copy()
    print(f"Total stations: {len(df)}")
    # 连接查询线路顺序
    res_df = df.merge(df_line[[
        'city_id', 'line_name_full', 'line_order', 'line_name_branch',
        'line_name_main', 'line_uid_k'
    ]],
                      how='left',
                      on=['city_id', 'line_uid_k'])
    res_df['line_name_full'] = res_df['line_name_full_y']
    print(f"Total stations after merge line info: {len(res_df)}")
    # 当len(res_df != len(df))时，说明有部分站点的line_name_full在df_line中没有匹配上
    if len(res_df) != len(df):
        print(
            f"WARNNING!, Stations with unmatched line info: {len(res_df) - len(df)}"
        )
        res_df_unmatched = res_df[res_df['line_name_full'].isna()]
        print(res_df_unmatched[[
            'city_id', 'city_name', 'line_name_full_x', 'st_name'
        ]])
        print(res_df.info())
    # 车站顺序
    res_df['st_order'] = res_df.groupby(['city_id', 'line_name_full'
                                         ]).cumcount() + 1
    # 车站id
    res_df['line_order'] = res_df['line_order'].astype(int)
    res_df['st_id_virtual'] = res_df.apply(
        lambda row:
        f"{row['city_id']}_{row['line_order']:03d}_{row['st_order']:03d}",
        axis=1)
    # 将st_name为空或字符串长度为0的站点st_name填充为st_id_virtual
    res_df['st_name'] = res_df['st_name'].replace('', np.nan)
    res_df['st_name'] = res_df['st_name'].fillna(res_df['st_id_virtual'])
    # 下一站id
    res_df['target_st_id_virtual'] = res_df.groupby(
        ['city_id', 'line_name_full'])['st_id_virtual'].shift(-1)
    # 如果line_is_loop为1，则最后一站的下一站为第一站
    # 找出所有环线线路
    loop_keys = res_df.loc[res_df['line_is_loop'] == 1,
                           ['city_id', 'line_name_full']].drop_duplicates()

    for _, row in loop_keys.iterrows():
        mask = (res_df['city_id'] == row['city_id']) & (
            res_df['line_name_full'] == row['line_name_full'])
        idx = res_df.loc[mask].index

        if len(idx) > 1:  # 至少要有两站才能构成环线
            first_idx = idx[0]
            last_idx = idx[-1]
            res_df.loc[last_idx,
                       'target_st_id_virtual'] = res_df.loc[first_idx,
                                                            'st_id_virtual']

    # 数据类型，需要转换为float
    cols_float = [
        'st_x', 'st_y', 'st_label_offset_x', 'st_label_offset_y', 'st_proj_x',
        'st_proj_y', 'line_label_x', 'line_label_y'
    ]
    for col in cols_float:
        res_df[col] = pd.to_numeric(res_df[col], errors='coerce')

    # 按target_st_id_virtual， 添加target_st_name, target_st_x, target_st_y
    res_df = res_df.merge(
        res_df[['st_id_virtual', 'st_name', 'st_x', 'st_y']].rename(
            columns={
                'st_id_virtual': 'target_st_id_virtual',
                'st_name': 'target_st_name',
                'st_x': 'target_st_x',
                'st_y': 'target_st_y'
            }),
        how='left',
        on='target_st_id_virtual')
    return res_df

#### 全量车站表
含虚拟车站，用于线条更柔和

In [15]:
dwd_subway_st_all =  get_dwd_subway_st(dwd_subway, dws_subway_bd_line, is_all_st=True)

Total stations: 18737
Total stations after merge line info: 18767
WARNNING!, Stations with unmatched line info: 30
Empty DataFrame
Columns: [city_id, city_name, line_name_full_x, st_name]
Index: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18767 entries, 0 to 18766
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   city_id                  18767 non-null  object 
 1   city_name                18767 non-null  object 
 2   city_name_e              18767 non-null  object 
 3   line_name_full_x         18767 non-null  object 
 4   line_name                18767 non-null  object 
 5   line_show_label          18767 non-null  int64  
 6   line_n                   18767 non-null  object 
 7   line_is_loop             18767 non-null  int64  
 8   line_label_x             18767 non-null  object 
 9   line_label_y             18767 non-null  object 
 10  line_label_rotation      18767 non-null 

#### 真实车站表
真实存在的， 即st_is_station == 1

In [16]:
dwd_subway_st_real =  get_dwd_subway_st(dwd_subway, dws_subway_bd_line, is_all_st=False)


Total stations: 13570
Total stations after merge line info: 13588
WARNNING!, Stations with unmatched line info: 18
Empty DataFrame
Columns: [city_id, city_name, line_name_full_x, st_name]
Index: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13588 entries, 0 to 13587
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   city_id                  13588 non-null  object 
 1   city_name                13588 non-null  object 
 2   city_name_e              13588 non-null  object 
 3   line_name_full_x         13588 non-null  object 
 4   line_name                13588 non-null  object 
 5   line_show_label          13588 non-null  int64  
 6   line_n                   13588 non-null  object 
 7   line_is_loop             13588 non-null  int64  
 8   line_label_x             13588 non-null  object 
 9   line_label_y             13588 non-null  object 
 10  line_label_rotation      13588 non-null 

In [17]:
dwd_subway_st_real

,city_id,city_name,city_name_e,line_name_full_x,line_name,line_show_label,line_n,line_is_loop,line_label_x,line_label_y,...,line_order,line_name_branch,line_name_main,line_name_full,st_order,st_id_virtual,target_st_id_virtual,target_st_name,target_st_x,target_st_y
0,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,1,1号线八通线,1号线八通线,地铁1号线八通线,1,131_001_001,131_001_002,八角游乐园,-410.0,220.0
1,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,1,1号线八通线,1号线八通线,地铁1号线八通线,2,131_001_002,131_001_003,八宝山,-320.0,220.0
2,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,1,1号线八通线,1号线八通线,地铁1号线八通线,3,131_001_003,131_001_004,玉泉路,-240.0,220.0
3,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,1,1号线八通线,1号线八通线,地铁1号线八通线,4,131_001_004,131_001_005,五棵松,-150.0,220.0
4,131,北京,beijing,地铁1号线八通线,1号线八通线,0,23,0,-498.9,139.1,...,1,1号线八通线,1号线八通线,地铁1号线八通线,5,131_001_005,131_001_006,万寿路,-60.0,220.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13583,268s,郑州s,zhengzhou,城郊线,城郊线,0,14,0,520.0,870.0,...,12,城郊线,城郊线,城郊线,13,268s_012_013,268s_012_014,3号航站楼,1960.0,1930.0
13584,268s,郑州s,zhengzhou,城郊线,城郊线,0,14,0,520.0,870.0,...,12,城郊线,城郊线,城郊线,14,268s_012_014,268s_012_015,机场东,2050.0,1960.0
13585,268s,郑州s,zhengzhou,城郊线,城郊线,0,14,0,520.0,870.0,...,12,城郊线,城郊线,城郊线,15,268s_012_015,268s_012_016,港区会展,2140.0,1990.0
13586,268s,郑州s,zhengzhou,城郊线,城郊线,0,14,0,520.0,870.0,...,12,城郊线,城郊线,城郊线,16,268s_012_016,268s_012_017,郑州航空港站,2230.0,2020.0


## ADS
ADS 层（Application Data Store，应用数据层）

面向应用和报表的最终数据，通常是宽表、指标表。

特点：与具体应用、报表或接口一一对应。

作用：满足业务方“即取即用”，保证查询效率。

1. city_infos 城市信息表
2. city_stats 城市维度统计指标
3. city_line_links 线路间换乘站数量， 不需要lineStyle数据
4. city_line_data

### 城市信息

#### dict

In [18]:
dwd_subway_city = get_city_infos_dws(dwd_subway, dwd_city_info)
dwd_subway_city

,city_id,city_name,line_count,city_order,city_name_e,province,province_e,country,country_e,note
0,60732,纽约,32,1,New York,纽约州,New York State,美国,United States,None
1,30016,首尔,30,2,Seoul,首尔特别市,Seoul Special City,韩国,South Korea,None
2,257,广州,29,3,Guangzhou,广东省,Guangdong Sheng,中国,China,None
3,131,北京,28,4,Beijing,北京市,Beijing Shi,中国,China,None
4,289,上海,25,5,Shanghai,上海市,Shanghai Shi,中国,China,None
...,...,...,...,...,...,...,...,...,...,...
75,155,许昌,1,76,Xuchang,河南省,Henan Sheng,中国,China,None
76,30007,光州,1,77,Gwangju,光州广域市,Gwangju Metropolitan City,韩国,South Korea,None
77,313,湘潭,1,78,Xiangtan,湖南省,Hunan Sheng,中国,China,None
78,122,鄂州,1,79,Ezhou,湖北省,Hubei Sheng,中国,China,None


In [19]:
def get_city_infos_dict(df):
    """
    构建城市相关信息字典，包括城市列表、国家列表、分组等
    df: dwd_subway_city数据
    """
    df['city_name'] = df['city_name'].str.replace('特别行政区', '')
    city_id_list = df['city_id'].tolist()
    city_name_list = df['city_name'].tolist()
    country_list = (
        df.groupby('country')
        .size()
        .sort_values(ascending=False)
        .index.tolist()
    )
    country_city_dict = {}
    for country, group in df.groupby('country'):
        country_city_dict[country] = group['city_id'].tolist()
    city_dict = {}
    for _, row in df.iterrows():
        city_dict[row['city_id']] = {
            'city_id': row['city_id'],
            'city_name': row['city_name'],
            'city_name_e': row['city_name_e'],
            'province': row['province'],
            'province_e': row['province_e'],
            'country': row['country'],
            'country_e': row['country_e'],
            'city_order': row['city_order']
        }
    city_infos = {
        'city_id_list': city_id_list,
        'city_name_list': city_name_list,
        'country_list': country_list,
        'country_city_dict': country_city_dict,
        'city_dict': city_dict
    }
    return city_infos

city_infos = get_city_infos_dict(dwd_subway_city)
city_infos

{'city_id_list': ['60732',
  '30016',
  '257',
  '131',
  '289',
  '75',
  '51271',
  '340',
  '49872',
  '51314',
  '132',
  '179',
  '26041',
  '315',
  '48552',
  '233',
  '65531',
  '268s',
  '39817',
  '268',
  '20001',
  '332',
  '218',
  '9002',
  '2912',
  '158',
  '224',
  '26033',
  '288',
  '236',
  '104',
  '138',
  '167',
  '180',
  '58',
  '30001',
  '300',
  '26001',
  '127',
  '20508',
  '53',
  '317',
  '261',
  '52390',
  '146',
  '163',
  '323',
  '316',
  '9019',
  '293',
  '48',
  '30004',
  '150',
  '119',
  '194',
  '26019',
  '39816',
  '53009',
  '36',
  '348',
  '92',
  '333',
  '321',
  '129',
  '153',
  '161',
  '176',
  '178',
  '197',
  '26022',
  '242',
  '244',
  '189',
  '2911',
  '30005',
  '155',
  '30007',
  '313',
  '122',
  '274'],
 'city_name_list': ['纽约',
  '首尔',
  '广州',
  '北京',
  '上海',
  '成都',
  '巴塞罗那',
  '深圳',
  '巴黎',
  '马德里',
  '重庆',
  '杭州',
  '东京',
  '南京',
  '伊斯坦布尔',
  '西安',
  '莫斯科',
  '郑州s',
  '圣保罗',
  '郑州',
  '新加坡',
  '天津',
  '武汉',
  '台北',


#### 保存本地json

In [20]:
# city_infos
city_infos = get_city_infos_dict(dwd_subway_city)
with open("json_data/city_infos.json", "w", encoding="utf-8") as f:
    json.dump(city_infos, f, ensure_ascii=False, indent=4)

### 城市指标


#### df

In [21]:
def get_city_stats_ads(df_st, df_city):
    """
    按城市统计地铁指标：线路数量、车站数量、换乘站数量及占比
    df_st: dwd_subway_st_real,计算城市指标时，使用真实车站数据
    df_city: dwd_subway_city数据
    """
    city_stats = df_st.groupby(['city_id', 'city_name']).agg(
        line_name_main_count=('line_name_main', 'nunique'),
        line_name_full_count=('line_name_full', 'nunique'),
        station_count=('st_name', 'nunique'),
        transfer_station_count=('st_name', lambda x: x[df_st.loc[x.index, 'st_is_exchange'] == 1].nunique()),
        loop_line_count=('line_name_main', lambda x: x[df_st.loc[x.index, 'line_is_loop'] == 1].nunique())
    ).reset_index()
    # 是否存在支线
    city_stats['has_branch_line'] = (city_stats['line_name_main_count'] < city_stats['line_name_full_count']).astype(int)
    # 计算换乘站占比，保留4位小数
    city_stats['transfer_station_ratio'] = city_stats['transfer_station_count'] / city_stats['station_count']
    city_stats['transfer_station_ratio'] = city_stats['transfer_station_ratio'].fillna(0).round(4)
    
    city_stats = city_stats.merge(
        df_city[['city_id', 'city_order', 'city_name_e']],
        how='left', on='city_id'
    ).sort_values('city_order').reset_index(drop=True)
    city_stats['city_name'] = city_stats['city_name'].str.replace('特别行政区', '')
    return city_stats

ads_subway_city_stats = get_city_stats_ads(dwd_subway_st_real, dwd_subway_city)
ads_subway_city_stats

,city_id,city_name,line_name_main_count,line_name_full_count,station_count,transfer_station_count,loop_line_count,has_branch_line,transfer_station_ratio,city_order,city_name_e
0,60732,纽约,26,32,463,212,0,1,0.4579,1,New York
1,30016,首尔,23,30,626,107,0,1,0.1709,2,Seoul
2,257,广州,28,32,455,105,1,1,0.2308,3,Guangzhou
3,131,北京,28,28,423,106,2,0,0.2506,4,Beijing
4,289,上海,22,25,433,105,1,1,0.2425,5,Shanghai
...,...,...,...,...,...,...,...,...,...,...,...
75,155,许昌,1,1,27,0,0,0,0.0000,76,Xuchang
76,30007,光州,1,1,20,0,0,0,0.0000,77,Gwangju
77,313,湘潭,1,1,33,7,0,0,0.2121,78,Xiangtan
78,122,鄂州,1,1,23,6,0,0,0.2609,79,Ezhou


#### dict

In [22]:
def get_city_stats_dict(df_city_stats):
    """
    构建城市地铁统计指标字典
    df_city_stats: ads_subway_city_stats数据
    """
    city_stats_dict = {}
    for _, row in df_city_stats.iterrows():
        city_stats_dict[row['city_id']] = {
            'city_id': row['city_id'],
            'city_name': row['city_name'],
            'city_name_e': row['city_name_e'],
            'line_name_main_count': int(row['line_name_main_count']),
            'line_name_full_count': int(row['line_name_full_count']),
            'has_branch_line': int(row['has_branch_line']),
            'station_count': int(row['station_count']),
            'transfer_station_count': int(row['transfer_station_count']),
            'transfer_station_ratio': float(row['transfer_station_ratio']),
            'city_order': int(row['city_order'])
        }
    return city_stats_dict
city_stats_dict = get_city_stats_dict(ads_subway_city_stats)
city_stats_dict

{'60732': {'city_id': '60732',
  'city_name': '纽约',
  'city_name_e': 'New York',
  'line_name_main_count': 26,
  'line_name_full_count': 32,
  'has_branch_line': 1,
  'station_count': 463,
  'transfer_station_count': 212,
  'transfer_station_ratio': 0.4579,
  'city_order': 1},
 '30016': {'city_id': '30016',
  'city_name': '首尔',
  'city_name_e': 'Seoul',
  'line_name_main_count': 23,
  'line_name_full_count': 30,
  'has_branch_line': 1,
  'station_count': 626,
  'transfer_station_count': 107,
  'transfer_station_ratio': 0.1709,
  'city_order': 2},
 '257': {'city_id': '257',
  'city_name': '广州',
  'city_name_e': 'Guangzhou',
  'line_name_main_count': 28,
  'line_name_full_count': 32,
  'has_branch_line': 1,
  'station_count': 455,
  'transfer_station_count': 105,
  'transfer_station_ratio': 0.2308,
  'city_order': 3},
 '131': {'city_id': '131',
  'city_name': '北京',
  'city_name_e': 'Beijing',
  'line_name_main_count': 28,
  'line_name_full_count': 28,
  'has_branch_line': 0,
  'station_c

#### 保存json

In [23]:
with open("json_data/city_stats.json", "w", encoding="utf-8") as f:
    json.dump(city_stats_dict, f, ensure_ascii=False, indent=4)

### 线路数据

#### df

In [24]:
# 线路换乘线网图比例
def get_chart_ratio(df_city, line_list=None):
    df_line = df_city[df_city['line_name_branch'].isin(line_list)] if line_list else df_city
    x_min = df_line['st_x'].min()
    x_max = df_line['st_x'].max()
    y_min = df_line['st_y'].min()
    y_max = df_line['st_y'].max()
    x_range = x_max - x_min
    y_range = y_max - y_min
    ratio = x_range / y_range if y_range != 0 else 1
    return round(float(ratio), 2)

# 线路换乘线网所有车站数量
def get_line_transfer_lines_st_count(df_city, line_list):
    df_line = df_city[df_city['line_name_branch'].isin(line_list)]
    return df_line['st_name'].nunique()

# 按照st_name提取所在线路的st_id_virtual
def get_line_st_ids(df_city, line_name, st_name_list):
    df_line = df_city[df_city['line_name_branch'] == line_name]
    return df_line[df_line['st_name'].isin(st_name_list)]['st_id_virtual'].unique().tolist()

In [25]:
def get_line_data_ads(df_st):
    """
    获取线路数据
    df_st: dwd_subway_st_real数据
    """
    city_ids = df_st['city_id'].unique().tolist()
    res_df = pd.DataFrame()
    for city_id in city_ids:
        df_city = df_st[df_st['city_id'] == city_id]
        df_city_line = df_city.groupby(['city_id', 'city_name', 'line_name_full', 'line_name', 'line_name_branch', 'line_name_main', 'line_color']).agg(
            line_order = ('line_order', 'first'),
            line_is_loop = ('line_is_loop', 'first'),
            st_count = ('st_name', 'nunique'),
            transfer_st_count = ('st_is_exchange', 'sum'),
            transfer_sts = ('st_name', lambda x: x[df_city.loc[x.index, 'st_is_exchange'] == 1].unique().tolist(),)
            ).reset_index()
        # transfer_sts_ids
        df_city_line['transfer_sts_ids'] = df_city_line.apply(
            lambda row: get_line_st_ids(df_city, row['line_name_branch'], row['transfer_sts']), axis=1
            )
        # 线路顺序
        line_sorted = df_city_line[['line_name_branch', 'line_order']].drop_duplicates().sort_values('line_order')['line_name_branch'].tolist()
        # 换乘线路，按transfer_sts确定车站所在路线，并将换乘线路列表添加到transfer_lines字段中
        df_city_line['transfer_lines_all'] = df_city_line['transfer_sts'].apply(
            lambda sts: df_city[df_city['st_name'].isin(sts)]['line_name_branch'].unique().tolist()
            )
        # 去重
        df_city_line['transfer_lines_all'] = df_city_line['transfer_lines_all'].apply(lambda x: list(set(x)))
        # 按line_sorted排序
        df_city_line['transfer_lines_all'] = df_city_line['transfer_lines_all'].apply(lambda x: sorted(x, key=lambda y: line_sorted.index(y)) if isinstance(x, list) else x)
        # 去除自身
        df_city_line['transfer_lines'] = df_city_line.apply(lambda row: [line for line in row['transfer_lines_all'] if line != row['line_name_branch']], axis=1)
        df_city_line['transfer_lines_count'] = df_city_line['transfer_lines'].apply(len)

        # 线路图比例
        df_city_line['chart_ratio'] = df_city_line['transfer_lines_all'].apply(lambda x: get_chart_ratio(df_city, x)).fillna(1.0)

        # 城市线路数
        df_city_line['city_line_count'] = len(df_city_line)
        # 城市车站数
        df_city_line['city_st_count'] = df_city['st_name'].nunique()
        # transfer_lines_all所有车站的数量，按st_name去重
        df_city_line['transfer_lines_all_count'] = df_city_line['transfer_lines_all'].apply(lambda x: get_line_transfer_lines_st_count(df_city, x))
        
        # 线路排序
        df_city_line = df_city_line.sort_values('line_order')
        res_df = pd.concat([res_df, df_city_line], ignore_index=True)
    return res_df

In [26]:
ads_subway_line_data = get_line_data_ads(dwd_subway_st_real)
ads_subway_line_data

,city_id,city_name,line_name_full,line_name,line_name_branch,line_name_main,line_color,line_order,line_is_loop,st_count,transfer_st_count,transfer_sts,transfer_sts_ids,transfer_lines_all,transfer_lines,transfer_lines_count,chart_ratio,city_line_count,city_st_count,transfer_lines_all_count
0,131,北京,地铁1号线八通线,1号线八通线,1号线八通线,1号线八通线,#D3746E,1,0,35,13,"[公主坟, 军事博物馆, 木樨地, 复兴门, 西单, 王府井, 东单, 建国门, 永安里, ...","[131_001_007, 131_001_008, 131_001_009, 131_00...","[1号线八通线, 2号线, 4号线大兴线, 5号线, 7号线, 8号线, 9号线, 10号线...","[2号线, 4号线大兴线, 5号线, 7号线, 8号线, 9号线, 10号线, 14号线, ...",10,0.93,28,423,268
1,131,北京,地铁2号线,2号线,2号线,2号线,#568BB2,2,1,18,13,"[西直门, 积水潭, 鼓楼大街, 雍和宫, 东直门, 东四十条, 朝阳门, 建国门, 崇文门...","[131_002_001, 131_002_002, 131_002_003, 131_00...","[1号线八通线, 2号线, 3号线, 4号线大兴线, 5号线, 6号线, 8号线, 13号线...","[1号线八通线, 3号线, 4号线大兴线, 5号线, 6号线, 8号线, 13号线, 19号...",9,1.19,28,423,197
2,131,北京,地铁3号线,3号线,3号线,3号线,#F05771,3,0,10,5,"[东四十条, 工人体育场, 团结湖, 朝阳公园, 东坝北]","[131_003_001, 131_003_002, 131_003_003, 131_00...","[2号线, 3号线, 10号线, 12号线, 14号线, 17号线]","[2号线, 10号线, 12号线, 14号线, 17号线]",5,0.88,28,423,133
3,131,北京,地铁4号线大兴线,4号线大兴线,4号线大兴线,4号线大兴线,#49ACBD,4,0,35,12,"[西苑, 海淀黄庄, 人民大学, 国家图书馆, 西直门, 平安里, 西单, 宣武门, 菜市口...","[131_004_003, 131_004_007, 131_004_008, 131_00...","[1号线八通线, 2号线, 4号线大兴线, 6号线, 7号线, 9号线, 10号线, 12号...","[1号线八通线, 2号线, 6号线, 7号线, 9号线, 10号线, 12号线, 13号线,...",11,1.19,28,423,265
4,131,北京,地铁5号线,5号线,5号线,5号线,#C162A1,5,0,23,13,"[天通苑, 立水桥, 大屯路东, 惠新西街南口, 和平西桥, 雍和宫, 北新桥, 东四, 东...","[131_005_002, 131_005_004, 131_005_007, 131_00...","[1号线八通线, 2号线, 5号线, 6号线, 7号线, 10号线, 12号线, 13号线,...","[1号线八通线, 2号线, 6号线, 7号线, 10号线, 12号线, 13号线, 14号线...",12,1.41,28,423,264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
648,268s,郑州s,地铁8号线,8号线,8号线,8号线,#E3F697,8,0,28,9,"[郑州大学, 同乐, 白庙, 东风路, 小营, 龙湖中环南, 高铁公园, 郑州东站, 圃田西]","[268s_008_004, 268s_008_010, 268s_008_012, 268...","[1号线, 2号线, 3号线, 4号线, 5号线, 6号线, 7号线, 8号线, 12号线]","[1号线, 2号线, 3号线, 4号线, 5号线, 6号线, 7号线, 12号线]",8,1.82,12,208,181
649,268s,郑州s,地铁10号线,10号线,10号线,10号线,#CD877F,9,0,12,5,"[市委党校, 市中心医院, 绿城广场, 医学院, 郑州火车站]","[268s_009_005, 268s_009_009, 268s_009_010, 268...","[1号线, 5号线, 7号线, 10号线, 14号线]","[1号线, 5号线, 7号线, 14号线]",4,1.47,12,208,89
650,268s,郑州s,地铁12号线,12号线,12号线,12号线,#578BEF,10,0,11,6,"[龙子湖, 高铁公园, 儿童医院·颐和医院, 黄河南路, 西周, 福塔东]","[268s_010_002, 268s_010_004, 268s_010_006, 268...","[1号线, 3号线, 5号线, 8号线, 12号线]","[1号线, 3号线, 5号线, 8号线]",4,2.36,12,208,111
651,268s,郑州s,地铁14号线,14号线,14号线,14号线,#C5A7CA,11,0,6,3,"[铁炉, 市委党校, 奥体中心]","[268s_011_002, 268s_011_004, 268s_011_005]","[1号线, 6号线, 10号线, 14号线]","[1号线, 6号线, 10号线]",3,1.91,12,208,69


#### dict

In [27]:
def get_line_data_dict(df_line_data):
    """
    构建线路数据字典
    df_line_data: ads_subway_line_data数据
    """
    city_ids = df_line_data['city_id'].unique().tolist()
    res = defaultdict(dict)
    for city_id in city_ids:
        df_city = df_line_data[df_line_data['city_id'] == city_id]
        res[city_id]['city_id'] = city_id
        res[city_id]['city_name'] = df_city['city_name'].iloc[0]
        res[city_id]['city_line_count'] = int(df_city['city_line_count'].iloc[0])
        res[city_id]['city_st_count'] = int(df_city['city_st_count'].iloc[0])
        lines_data = []
        for _, row in df_city.iterrows():
            lines_data.append({
                'line_name_full': row['line_name_full'],
                'line_name': row['line_name'],
                'line_name_branch': row['line_name_branch'],
                'line_name_main': row['line_name_main'],
                'line_color': row['line_color'],
                'line_order': int(row['line_order']),
                'st_count': int(row['st_count']),
                'transfer_st_count': int(row['transfer_st_count']),
                'transfer_sts': row['transfer_sts'],
                'transfer_sts_ids': row['transfer_sts_ids'],
                'transfer_lines_all': row['transfer_lines_all'],
                'transfer_lines_all_count': int(row['transfer_lines_all_count']),
                'transfer_lines': row['transfer_lines'],
                'transfer_lines_count': int(row['transfer_lines_count']),
                'chart_ratio': float(row['chart_ratio'])
            })
        res[city_id]['lines_data'] = lines_data
    return res
line_data_dict = get_line_data_dict(ads_subway_line_data)
line_data_dict

defaultdict(dict,
            {'131': {'city_id': '131',
              'city_name': '北京',
              'city_line_count': 28,
              'city_st_count': 423,
              'lines_data': [{'line_name_full': '地铁1号线八通线',
                'line_name': '1号线八通线',
                'line_name_branch': '1号线八通线',
                'line_name_main': '1号线八通线',
                'line_color': '#D3746E',
                'line_order': 1,
                'st_count': 35,
                'transfer_st_count': 13,
                'transfer_sts': ['公主坟',
                 '军事博物馆',
                 '木樨地',
                 '复兴门',
                 '西单',
                 '王府井',
                 '东单',
                 '建国门',
                 '永安里',
                 '国贸',
                 '大望路',
                 '花庄',
                 '环球度假区'],
                'transfer_sts_ids': ['131_001_007',
                 '131_001_008',
                 '131_001_009',
                 '131_001_011',
                 '131_00

#### 保存json

In [28]:
with open("json_data/line_data.json", "w", encoding="utf-8") as f:
    json.dump(line_data_dict, f, ensure_ascii=False, indent=4)

### 线路间换乘站数量矩阵

#### df

In [29]:
def get_line_transfer_matrix(df_city):
    # 只保留换乘站
    df_city = df_city[df_city['st_is_exchange'] == 1]
    # 获取所有线路名
    lines = df_city.sort_values(by='line_order')['line_name_branch'].unique()

    # 构建线路间换乘站数量的矩阵
    # 统计每个车站涉及的线路数
    station_line_counts = df_city.groupby('st_name')['line_name_branch'].nunique()
    # 换乘站定义为涉及多条线路的车站
    transfer_stations = station_line_counts[station_line_counts > 1]
    transfer_station_set = set(transfer_stations.index)
    line_transfer = pd.DataFrame(0, index=lines, columns=lines)

    # 主线的支线，如果num_common>1， 则默认为1

    for line1 in lines:
        stations1 = set(df_city[df_city['line_name_branch'] == line1]['st_name']) & transfer_station_set
        line1_main = df_city[df_city['line_name_branch'] == line1]['line_name_main'].iloc[0]
        for line2 in lines:
            line2_main = df_city[df_city['line_name_branch'] == line2]['line_name_main'].iloc[0]
            if line1 == line2:
                continue
            stations2 = set(df_city[df_city['line_name_branch'] == line2]['st_name']) & transfer_station_set
            # 两条线路的换乘站交集数量
            num_common = len(stations1 & stations2)
            if line1_main == line2_main and num_common > 1:
                num_common = 1 
            line_transfer.loc[line1, line2] = num_common
    # 新建index列,并将原index列移动到第一列,然后重命名为line_name_branch
    line_transfer = line_transfer.reset_index().rename(columns={'index': 'line_name_branch'})

    return line_transfer

In [30]:
df_city = dwd_subway_st_real[dwd_subway_st_real['city_id'] == '75']
df_city

,city_id,city_name,city_name_e,line_name_full_x,line_name,line_show_label,line_n,line_is_loop,line_label_x,line_label_y,...,line_order,line_name_branch,line_name_main,line_name_full,st_order,st_id_virtual,target_st_id_virtual,target_st_name,target_st_x,target_st_y
3134,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),1,75_001_001,75_001_002,升仙湖,60.0,-860.0
3135,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),2,75_001_002,75_001_003,火车北站,-70.0,-720.0
3136,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),3,75_001_003,75_001_004,人民北路,-120.0,-610.0
3137,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),4,75_001_004,75_001_005,文殊院,-120.0,-540.0
3138,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),5,75_001_005,75_001_006,骡马市,-120.0,-390.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3639,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),2,75_020_002,75_020_003,幸福大道,870.0,1340.0
3640,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),3,75_020_003,75_020_004,苌弘广场,950.0,1340.0
3641,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),4,75_020_004,75_020_005,宝台,1030.0,1340.0
3642,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),5,75_020_005,75_020_006,资阳北站,1110.0,1340.0


In [31]:
line_transfer_matrix = get_line_transfer_matrix(df_city)
line_transfer_matrix

,line_name_branch,1号线(五根松-韦家碾),1号线(科学城-韦家碾),2号线,3号线,4号线,5号线,6号线,7号线,8号线,...,10号线,13号线,17号线,18号线,19号线,27号线,30号线,有轨电车蓉2线(成都西站-郫县西站),有轨电车蓉2线(新业路-仁和),S3线(资阳线)
0,1号线(五根松-韦家碾),0,1,1,1,1,0,1,2,1,...,0,1,0,3,0,1,0,0,0,0
1,1号线(科学城-韦家碾),1,0,1,1,1,0,2,2,1,...,0,1,0,5,0,1,0,0,0,0
2,2号线,1,1,0,1,1,1,2,2,1,...,0,0,1,0,0,1,1,0,1,0
3,3号线,1,1,1,0,1,1,1,2,0,...,4,1,1,0,1,0,0,0,0,0
4,4号线,1,1,1,1,0,1,1,2,1,...,0,0,0,0,1,0,0,1,0,0
5,5号线,0,0,1,1,1,0,2,2,1,...,1,1,1,0,1,1,1,0,0,0
6,6号线,1,2,2,1,1,2,0,2,1,...,0,1,0,1,1,1,1,1,0,0
7,7号线,2,2,2,2,2,2,2,0,2,...,1,2,2,1,0,0,0,0,0,0
8,8号线,1,1,1,0,1,1,1,2,0,...,0,1,0,0,1,0,1,0,0,0
9,9号线,1,1,0,1,1,1,1,0,1,...,1,1,1,1,0,0,0,1,0,0


#### dict

In [32]:
def get_line_transfer_dict(df_st_real):
	"""
	获取线路换乘数据
	df_st_real: dwd_subway_st_real数据
	"""
	city_ids = df_st_real['city_id'].unique().tolist()
	res = defaultdict(dict)
	for city_id in city_ids:
		df_city = df_st_real[df_st_real['city_id'] == city_id]
		df_matrix = get_line_transfer_matrix(df_city)
		df = df_matrix.set_index('line_name_branch')
		line_name = df.index.tolist()
		col_name = df.columns.tolist()
		res_dict = []
		res_list = []
		for r in range(df.shape[0]):
			for c in range(df.shape[1]):
				v = df.iloc[r, c]
				if v != 0:
					res_dict_i = {
						'source': line_name[r],
						'target': col_name[c],
						'value': int(v)
					}
					res_list_i = [line_name[r], col_name[c], int(v)]
					res_dict.append(res_dict_i)
					res_list.append(res_list_i)
		res[city_id] = {
			'city_id': city_id,
			'city_name': df_city['city_name'].iloc[0],
			'line_links': res_dict,
			'line_transfer_matrix': {
				'data': res_list,
				'lines': line_name
			}
		}
	return res
get_line_transfer_dict(dwd_subway_st_real)

defaultdict(dict,
            {'131': {'city_id': '131',
              'city_name': '北京',
              'line_links': [{'source': '1号线八通线', 'target': '2号线', 'value': 2},
               {'source': '1号线八通线', 'target': '4号线大兴线', 'value': 1},
               {'source': '1号线八通线', 'target': '5号线', 'value': 1},
               {'source': '1号线八通线', 'target': '7号线', 'value': 2},
               {'source': '1号线八通线', 'target': '8号线', 'value': 1},
               {'source': '1号线八通线', 'target': '9号线', 'value': 1},
               {'source': '1号线八通线', 'target': '10号线', 'value': 2},
               {'source': '1号线八通线', 'target': '14号线', 'value': 1},
               {'source': '1号线八通线', 'target': '16号线', 'value': 1},
               {'source': '1号线八通线', 'target': '17号线', 'value': 1},
               {'source': '2号线', 'target': '1号线八通线', 'value': 2},
               {'source': '2号线', 'target': '3号线', 'value': 1},
               {'source': '2号线', 'target': '4号线大兴线', 'value': 2},
               {'source': '2号线', '

#### 保存json

In [33]:
line_transfer_dict = get_line_transfer_dict(dwd_subway_st_real)
with open("json_data/line_transfer.json", "w", encoding="utf-8") as f:
    json.dump(line_transfer_dict, f, ensure_ascii=False, indent=4)

### 换乘数量topN

In [34]:
def get_transer_stats(df_st_real):
    """
    获取换乘统计数据
    df_st_real: dwd_subway_st_real数据
    1. 每个城市换乘线路数最多的10条线路
    2. 每个城市换乘车站数最多的10条线路
    3. 每个城市换乘车站邻接车站数最多的10个车站
    """
    city_ids = df_st_real['city_id'].unique().tolist()
    res = defaultdict(dict)
    for city_id in city_ids:
        df_city = df_st_real[df_st_real['city_id'] == city_id]
        line_df = get_line_data_ads(df_city)
        res_i = defaultdict(dict)
        res_i['city_id'] = line_df['city_id'].iloc[0]
        res_i['city_name'] = line_df['city_name'].iloc[0]
        # 线路换乘数据统计
        line_df = line_df[['line_name_branch', 'transfer_lines_count', 'transfer_st_count']]
        # 按transfer_lines_count降序
        line_df = line_df.sort_values(by=['transfer_lines_count', 'transfer_st_count'], ascending=False).reset_index(drop=True)
        res_i['line_transfer_line_count'] = {
            'lines': line_df['line_name_branch'].tolist()[:10],
            'line_counts': line_df['transfer_lines_count'].tolist()[:10],
            'st_counts': line_df['transfer_st_count'].tolist()[:10]
        }
        # 按transfer_st_count降序
        line_df = line_df.sort_values(by=['transfer_st_count', 'transfer_lines_count'], ascending=False).reset_index(drop=True)
        res_i['line_transfer_st_count'] = {
            'lines': line_df['line_name_branch'].tolist()[:10],
            'line_counts': line_df['transfer_lines_count'].tolist()[:10],
            'st_counts': line_df['transfer_st_count'].tolist()[:10]
        }
        # 车站邻接车站数，与车站换乘线路数
        st_df = df_city[['st_name', 'target_st_name']].dropna()
        G_i = nx.from_pandas_edgelist(st_df, source='st_name', target='target_st_name', create_using=nx.Graph())
        # 度数最大的10个节点
        degrees = dict(G_i.degree())
        # top_10_degrees = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]
        st_transfer_lines_count = df_city[df_city['st_is_exchange'] == 1].groupby('st_name')['line_name_branch'].nunique().reset_index().rename(columns={'line_name_branch': 'transfer_lines_count'})   
        # 将degrees数值加入st_transfer_lines_count中
        st_transfer_lines_count['st_count'] = st_transfer_lines_count['st_name'].map(degrees)
        # 按st_count降序后，再按line_name_branch降序
        st_transfer_lines_count = st_transfer_lines_count.sort_values(by=['st_count', 'transfer_lines_count'], ascending=False).reset_index(drop=True)

        st_names =[]
        trans_lines_count = []
        st_degree = []
        for _, row in st_transfer_lines_count.head(10).iterrows():
            st_names.append(row['st_name']) 
            trans_lines_count.append(int(row['transfer_lines_count']))
            st_degree.append(int(row['st_count']))
        # for st_name, degree in top_10_degrees:
        #     transfer_lines_count = st_transfer_lines_count.get(st_name, 0)
        #     st_names.append(st_name)
        #     trans_lines_count.append(transfer_lines_count)
        #     st_degree.append(degree)
        res_i['st_transfer_lines_count'] = {
            'st_names': st_names,
            'transfer_lines_counts': trans_lines_count,
            'degrees': st_degree
        }
        res[city_id] = res_i

    return res

In [35]:
transer_stats = get_transer_stats(dwd_subway_st_real)
with open("json_data/transfer_stats.json", "w", encoding="utf-8") as f:
    json.dump(transer_stats, f, ensure_ascii=False, indent=4)


### 图数据, 使用全量车站表

In [36]:
df_city = dwd_subway_st_all[dwd_subway_st_all['city_id'] == '75']
df_city

,city_id,city_name,city_name_e,line_name_full_x,line_name,line_show_label,line_n,line_is_loop,line_label_x,line_label_y,...,line_order,line_name_branch,line_name_main,line_name_full,st_order,st_id_virtual,target_st_id_virtual,target_st_name,target_st_x,target_st_y
4930,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),1,75_001_001,75_001_002,升仙湖,60.0,-860.0
4931,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),2,75_001_002,75_001_003,火车北站,-70.0,-720.0
4932,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),3,75_001_003,75_001_004,75_001_004,-110.0,-680.0
4933,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),4,75_001_004,75_001_005,75_001_005,-116.0,-670.0
4934,75,成都,chengdu,地铁1号线(五根松-韦家碾),1号线(五根松-韦家碾),0,16,0,-85.0,420.0,...,1,1号线(五根松-韦家碾),1号线,地铁1号线(五根松-韦家碾),5,75_001_005,75_001_006,75_001_006,-120.0,-660.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5650,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),3,75_020_003,75_020_004,幸福大道,870.0,1340.0
5651,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),4,75_020_004,75_020_005,苌弘广场,950.0,1340.0
5652,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),5,75_020_005,75_020_006,宝台,1030.0,1340.0
5653,75,成都,chengdu,地铁S3线(资阳线),地铁S3线(资阳线),0,None,0,900.0,1380.0,...,20,S3线(资阳线),S3线(资阳线),地铁S3线(资阳线),6,75_020_006,75_020_007,资阳北站,1110.0,1340.0


In [37]:
def get_city_graph_dict(df_city):
    """
    获取城市地铁图数据
    df_city: dwd_subway_st_real数据，按城市筛选后的数据
    """
    # 线路数据
    df_line = df_city[['line_name_branch', 'line_order', 'line_color', 'line_label_x', 'line_label_y']].drop_duplicates().sort_values('line_order')
    lines_list = df_line['line_name_branch'].tolist()
    lines_dict = defaultdict(dict)
    categories = []
    for idx, row in df_line.iterrows():
        lines_dict[row['line_name_branch']] = {
            'line_order': int(row['line_order']),
            'line_color': row['line_color'],
            'line_label_x': row['line_label_x'],
            'line_label_y': row['line_label_y'],
            'line_label': row['line_name_branch']
        }
        categories.append({'name': row['line_name_branch'], 
                           'itemStyle': {'color': row['line_color']}})
    # 线路节点
    line_st_dict = defaultdict(list)
    for line in lines_list:
        st_list = df_city[df_city['line_name_branch'] == line]['st_id_virtual'].tolist()
        line_st_dict[line] = st_list

    G_city = nx.from_pandas_edgelist(df_city[['st_name', 'target_st_name']].dropna(), source='st_name', target='target_st_name', create_using=nx.Graph())
    # 节点的度
    degrees = dict(G_city.degree())
    # 填充NaN值为0
    df_city = df_city.copy()
    df_city.fillna({'st_label_offset_x': 0, 'st_label_offset_y': 0}, inplace=True)
    df_city['label_offset_x'] = df_city['st_label_offset_x']
    df_city['label_offset_y'] = df_city['st_label_offset_y']

    # 车站节点
    nodes_data = []
    nodes_dict = defaultdict(dict)
    nodes_list = []
    for _, row in df_city.iterrows():
        node = {
            'id': row['st_id_virtual'],
            'name': row['st_name'],
            'x': row['st_x'],
            'y': row['st_y'],
            'label_offset_x': row['st_label_offset_x'],
            'label_offset_y': row['st_label_offset_y'],
            "symbolSize": degrees.get(row['st_name'], 1),
            "category": row['line_order'] - 1,
            "value": degrees.get(row['st_name'], 1),
            'is_exchange': bool(row['st_is_exchange']),
            "itemStyle": {
                            "color": row['line_color']},
            "is_station": bool(row['st_is_station']),
            'line_name_main': row['line_name_main'],
            'line_name_branch': row['line_name_branch'],
            'line_color': row['line_color']
        }
        nodes_dict[row['st_id_virtual']] = node
        nodes_data.append(node)
        nodes_list.append(row['st_id_virtual'])
    nodes_list = list(set(nodes_list))
    
    # 线路边
    df_edges = df_city[['st_id_virtual', 'target_st_id_virtual', 'line_name_branch', 'line_name_main', 'line_color']].dropna()
    edges = []
    for _, row in df_edges.iterrows():
        edge = {
            'source': row['st_id_virtual'],
            'target': row['target_st_id_virtual'],
            'line_name_branch': row['line_name_branch'],
            'line_color': row['line_color'],
            'lineStyle': {
                'color': row['line_color']
            }}
        edges.append(edge)
    
    # 图比例
    chart_ration = get_chart_ratio(df_city)
    
    city_graph = {
        'city_id': df_city['city_id'].iloc[0],
        'city_name': df_city['city_name'].iloc[0],
        'graph_data': {
            'lines': lines_list,
            'lines_dict': lines_dict,
            'line_st_dict': line_st_dict,
            'categories': categories,
            'nodes': nodes_data,
            'nodes_dict': nodes_dict,
            'nodes_list': nodes_list,
            'links': edges,
            'chart_ratio': chart_ration
    }}
    return city_graph

In [38]:
city_graph_dict = defaultdict(dict)
city_ids = dwd_subway_st_all['city_id'].unique().tolist()
for city_id in city_ids:
    df_city = dwd_subway_st_all[dwd_subway_st_all['city_id'] == city_id]
    city_graph = get_city_graph_dict(df_city)
    city_graph_dict[city_id] = city_graph
with open(f"json_data/city_graph.json", "w", encoding="utf-8") as f:
    json.dump(city_graph_dict, f, ensure_ascii=False, indent=4)
print(f"City graph data saved to city_graph.json")

City graph data saved to city_graph.json


## 概述文案
{城市名} 地铁目前共运营 {线路数量} 条线路，覆盖 {车站数量} 座车站，其中 换乘站 {换乘站数量} 座。
网络结构以 {网络形态，如“环线+放射状”/“多环多放射”/“格网型”} 为主，形成 {单中心/多中心} 换乘格局。
主要换乘枢纽包括 {典型换乘站 1、2、3}，分别连接 {X 条线路}，承担了核心换乘功能。
整体来看，{网络连通性情况，如“整体连通性较强，绝大部分线路相互贯通，个别支线通过换乘枢纽衔接”}。
在结构性指标方面，网络表现出 {平均换乘度、最大换乘度等特点，可以加上路径长度、网络直径等补充}，反映出 {地铁网络在城市交通体系中的作用，比如“强中心辐射特征”或“均衡覆盖”}。

In [50]:
# 使用真实车站数据作为基础数据
city_id = "127"
dwd_subway_st_real =  get_dwd_subway_st(dwd_subway, dws_subway_bd_line, is_all_st=False)
df = dwd_subway_st_real.copy()
df_city = df[df['city_id'] == city_id]
df_city

Total stations: 13570
Total stations after merge line info: 13588
WARNNING!, Stations with unmatched line info: 18
Empty DataFrame
Columns: [city_id, city_name, line_name_full_x, st_name]
Index: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13588 entries, 0 to 13587
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   city_id                  13588 non-null  object 
 1   city_name                13588 non-null  object 
 2   city_name_e              13588 non-null  object 
 3   line_name_full_x         13588 non-null  object 
 4   line_name                13588 non-null  object 
 5   line_show_label          13588 non-null  int64  
 6   line_n                   13588 non-null  object 
 7   line_is_loop             13588 non-null  int64  
 8   line_label_x             13588 non-null  object 
 9   line_label_y             13588 non-null  object 
 10  line_label_rotation      13588 non-null 

,city_id,city_name,city_name_e,line_name_full_x,line_name,line_show_label,line_n,line_is_loop,line_label_x,line_label_y,...,line_order,line_name_branch,line_name_main,line_name_full,st_order,st_id_virtual,target_st_id_virtual,target_st_name,target_st_x,target_st_y
7081,127,合肥,hefei,轨道交通1号线,1号线,0,23,0,60.0,-310.0,...,1,1号线,1号线,轨道交通1号线,1,127_001_001,127_001_002,兴华苑,60.0,-150.0
7082,127,合肥,hefei,轨道交通1号线,1号线,0,23,0,60.0,-310.0,...,1,1号线,1号线,轨道交通1号线,2,127_001_002,127_001_003,瑶海公园,60.0,-60.0
7083,127,合肥,hefei,轨道交通1号线,1号线,0,23,0,60.0,-310.0,...,1,1号线,1号线,轨道交通1号线,3,127_001_003,127_001_004,合肥火车站,60.0,40.0
7084,127,合肥,hefei,轨道交通1号线,1号线,0,23,0,60.0,-310.0,...,1,1号线,1号线,轨道交通1号线,4,127_001_004,127_001_005,长淮,60.0,140.0
7085,127,合肥,hefei,轨道交通1号线,1号线,0,23,0,60.0,-310.0,...,1,1号线,1号线,轨道交通1号线,5,127_001_005,127_001_006,明光路,60.0,220.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7266,127,合肥,hefei,轨道交通6号线,6号线,0,0,0,1210.0,750.0,...,6,6号线,6号线,轨道交通6号线,18,127_006_018,127_006_019,钢红,820.0,690.0
7267,127,合肥,hefei,轨道交通6号线,6号线,0,0,0,1210.0,750.0,...,6,6号线,6号线,轨道交通6号线,19,127_006_019,127_006_020,大兴集,950.0,690.0
7268,127,合肥,hefei,轨道交通6号线,6号线,0,0,0,1210.0,750.0,...,6,6号线,6号线,轨道交通6号线,20,127_006_020,127_006_021,伏龙,1080.0,690.0
7269,127,合肥,hefei,轨道交通6号线,6号线,0,0,0,1210.0,750.0,...,6,6号线,6号线,轨道交通6号线,21,127_006_021,127_006_022,龙塘,1210.0,690.0


In [51]:
df_city[df_city['line_is_loop'] == 1]

,city_id,city_name,city_name_e,line_name_full_x,line_name,line_show_label,line_n,line_is_loop,line_label_x,line_label_y,...,line_order,line_name_branch,line_name_main,line_name_full,st_order,st_id_virtual,target_st_id_virtual,target_st_name,target_st_x,target_st_y


In [52]:
dws_subway_city = get_city_infos_dws(dwd_subway, dwd_city_info)
ads_subway_city_stats = get_city_stats_ads(dwd_subway_st_real, dws_subway_city)
df_city_stats = ads_subway_city_stats[ads_subway_city_stats['city_id'] == city_id]
df_city_stats

,city_id,city_name,line_name_main_count,line_name_full_count,station_count,transfer_station_count,loop_line_count,has_branch_line,transfer_station_ratio,city_order,city_name_e
38,127,合肥,6,6,174,17,0,0,0.0977,39,Hefei


In [53]:
transer_stats = get_transer_stats(dwd_subway_st_real)
city_transer_stats = transer_stats[city_id]['st_transfer_lines_count']
city_transer_stats

{'st_names': ['合肥南站',
  '三孝口',
  '东七里',
  '北雁湖',
  '合肥火车站',
  '图书馆',
  '大东门',
  '尧渡河路',
  '市第三医院',
  '方庙'],
 'transfer_lines_counts': [3, 2, 2, 2, 2, 2, 2, 2, 2, 2],
 'degrees': [6, 4, 4, 4, 4, 4, 4, 4, 4, 4]}

In [54]:
def analyze_city_subway(df_city, df_city_stats, city_transer_stats):
    """
    输入城市地铁数据（包括 station_id, line_name, line_is_loop, edges 等字段），
    输出结构化描述。
    """
    city_df = df_city.copy()
    city_stats = df_city_stats.copy()

    # === 基础指标 ===
    city_name = city_stats['city_name'].iloc[0]
    line_count = int(city_stats['line_name_main_count'].iloc[0])
    line_bnranch_count = int(city_stats['line_name_full_count'].iloc[0])
    has_branch_line = bool(city_stats['has_branch_line'].iloc[0])
    station_count = int(city_stats['station_count'].iloc[0])
    
    # === 换乘站统计 ===
    transfer_count = int(city_stats['transfer_station_count'].iloc[0])
    transfer_station_ratio = city_stats['transfer_station_ratio'].iloc[0]

    # === 主要换乘枢纽 ===
    city_transer_stats_filtered = {
        'st_names': [],
        'transfer_lines_counts': [],
        'degrees': []
    }
    for i, v in enumerate(city_transer_stats['transfer_lines_counts']):
        if v > 1:
            city_transer_stats_filtered['st_names'].append(city_transer_stats['st_names'][i])
            city_transer_stats_filtered['transfer_lines_counts'].append(v)
            city_transer_stats_filtered['degrees'].append(city_transer_stats['degrees'][i])
    top_hubs = city_transer_stats_filtered
    hub_names = top_hubs['st_names']
    hub_lines = top_hubs['transfer_lines_counts']
    hub_degrees = top_hubs['degrees']

    # === 环线数量 ===
    # loop_count = city_df.drop_duplicates(subset=["line_name"])["line_is_loop"].sum()
    loop_count = int(city_stats['loop_line_count'].iloc[0])
    loop_line_name = city_df[city_df['line_is_loop'] == 1]['line_name_main'].unique().tolist()

    # === 判断网络形态 ===
    if loop_count >= 2:
        network_shape = "多环多放射"
    elif loop_count == 1:
        network_shape = "环线+放射状"
    else:
        # 判断是否更接近格网型（通过平均度近似）
        # 建图
        edges = city_df[["st_name", "target_st_name"]].dropna().values.tolist()
        G = nx.Graph()
        G.add_edges_from(edges)
        avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
        network_shape = "格网型" if avg_degree >= 3 else "放射状"

    # === 中心格局判断 ===
    if transfer_count == 0:
        center_pattern = "暂无换乘"
    elif transfer_count <= 3:
        center_pattern = "单中心"
    else:
        center_pattern = "多中心"

    # === 网络连通性分析 ===
    G = nx.Graph()
    G.add_edges_from(city_df[["st_name", "target_st_name"]].dropna().values.tolist())
    components = nx.number_connected_components(G)
    if components == 1:
        connectivity = "整体连通性较强，绝大部分线路相互贯通"
    else:
        connectivity = f"网络存在{components}个子连通成分，部分支线需通过换乘枢纽衔接"

    # === 结构性指标 ===
    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    max_degree = max(dict(G.degree()).values())
    diameter = nx.diameter(G) if nx.is_connected(G) else None
    avg_path_length = nx.average_shortest_path_length(G) if nx.is_connected(G) else None

    # === 指标解读 ===
    structure_comment = (
        f"平均换乘度约为{avg_degree:.2f}，最大换乘度为{max_degree}。"
    )
    if diameter and avg_path_length:
        structure_comment += f"网络直径约为{diameter}，平均最短路径长度约为{avg_path_length:.2f}，"
    structure_comment += "反映出网络的层级放射性与较强的中心辐射特征。"

    # === 生成文字 ===
    branch_line_text = f"（支线单独计算共{line_bnranch_count}条）" if has_branch_line else ""
    loop_text = f"{city_name}地铁存在{loop_count}条环线（{'、'.join(loop_line_name)}），" if loop_count > 0 else ""
    desc = (
        f"{city_name}地铁目前共运营{line_count}条线路{branch_line_text}，覆盖{station_count}座车站，"
        f"其中换乘站{transfer_count}座，占比约{transfer_station_ratio:.0%}。"
        f"{loop_text}网络结构以{network_shape}为主，形成{center_pattern}换乘格局。"
    )

    if hub_names:
        # desc += (
        #     f"主要换乘枢纽包括 { '、'.join(hub_names[:3]) }等，"
        #     f"分别连接 { '、'.join(str(x) + ' 条线路' for x in hub_lines[:3]) }，"
        #     "承担了核心换乘功能。"
        # )
        desc += "主要换乘枢纽包括："
        # for i in range(3):
            # if hub_lines[i] == hub_lines[i+1] and hub_degrees[i] == hub_degrees[i+1]:
                  
            #     desc += f"{hub_names[i]}（连接{hub_lines[i]}条线路，邻接{hub_degrees[i]}个车站）；"
        merged_hubs = []
        for i in range(3):
            if i > 0 and hub_lines[i] == hub_lines[i - 1] and hub_degrees[i] == hub_degrees[i - 1]:
             merged_hubs[-1]['stations'].append(hub_names[i])
            else:
             merged_hubs.append({'stations': [hub_names[i]], 'lines': hub_lines[i], 'degrees': hub_degrees[i]})

        for hub in merged_hubs:
            stations = '、'.join(hub['stations'])
            desc += f"{stations}（连接{hub['lines']}条线路，邻接{hub['degrees']}个车站）；"
        desc = desc.rstrip('；') + f"，以及{'、'.join(hub_names[3:])}等换乘站共同承担了核心换乘功能。"

    desc += (
        f"整体来看，{city_name}地铁网络{connectivity}。"
        f"在结构性指标方面，{structure_comment}"
    )

    return desc


# === 运行示例 ===
print(analyze_city_subway(df_city, df_city_stats, city_transer_stats))


合肥地铁目前共运营6条线路，覆盖174座车站，其中换乘站17座，占比约10%。网络结构以放射状为主，形成多中心换乘格局。主要换乘枢纽包括：合肥南站（连接3条线路，邻接6个车站）；三孝口、东七里（连接2条线路，邻接4个车站），以及北雁湖、合肥火车站、图书馆、大东门、尧渡河路、市第三医院、方庙等换乘站共同承担了核心换乘功能。整体来看，合肥地铁网络整体连通性较强，绝大部分线路相互贯通。在结构性指标方面，平均换乘度约为2.11，最大换乘度为6。网络直径约为42，平均最短路径长度约为14.10，反映出网络的层级放射性与较强的中心辐射特征。


In [44]:
723/3*4

964.0